# Step 1
- Install the libraries

In [57]:
pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-classic chromadb pypdf python-dotenv rank-bm25 -q

Note: you may need to restart the kernel to use updated packages.


# Step 2 
- Verify installation

In [58]:
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"✅ LangChain version: {langchain.__version__}")
    print("✅ All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")

Python version: 3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
✅ LangChain version: 1.2.12
✅ All core packages installed successfully!


# Step 3 
- Library imports, set the '.env' variable and API key

In [59]:
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"✅ Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("✅ OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("✅ API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\n✅ All imports successful!")
print("📚 Using modern LangChain patterns: create_agent with @tool decorator")


✅ Loading .env from: /Users/vaisakh/Desktop/Learnings/ai-engineering-bootcamp/rag-powered-q&a-system/.env
✅ OpenAI API key loaded successfully!
✅ API key validated successfully!

✅ All imports successful!
📚 Using modern LangChain patterns: create_agent with @tool decorator


# Step 4 
- Loading the documents !!! Load the data from the PDF file(s)

In [60]:
import os
import logging

from langchain_community.document_loaders import PyPDFLoader

# Define the folder path where your PDFs are stored
pdf_folder_path = "./documents" 

# Initialize an empty list to store all documents
all_documents = []

# Iterate over all files in the directory
logging.getLogger("pypdf").setLevel(logging.ERROR)
for filename in os.listdir(pdf_folder_path):
    if filename.endswith(".pdf"):
        # Construct the full file path
        file_path = os.path.join(pdf_folder_path, filename)
        
        # Create a PyPDFLoader for the specific file
        loader = PyPDFLoader(file_path)
        
        # Load the documents from the PDF and add them to the list
        # .load() returns a list of Document objects (one per page)
        documents = loader.load()
        all_documents.extend(documents)

# You can now work with the combined list of documents
print(f"Successfully loaded {len(all_documents)} pages from all PDFs.")

# Example: Print metadata for the first few documents
for i, doc in enumerate(all_documents[:3]):
    print(f"Document {i+1} source: {doc.metadata['source']}, page: {doc.metadata['page']}")


Successfully loaded 111 pages from all PDFs.
Document 1 source: ./documents/nw_12.5.1.0_ueba_user_guide.pdf, page: 0
Document 2 source: ./documents/nw_12.5.1.0_ueba_user_guide.pdf, page: 1
Document 3 source: ./documents/nw_12.5.1.0_ueba_user_guide.pdf, page: 2


# Step 5 
- Split the documents into chunks

In [61]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True # Helps in tracking exactly where the chunk started
)

all_splits = text_splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} pages into {len(all_splits)} chunks.")
print(f"\n📊 Chunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in all_splits) / len(all_splits):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in all_splits)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in all_splits)} characters")

print(f"\n📄 First chunk preview:")
print("-" * 60)
print(all_splits[0].page_content)
print("-" * 60)

Split 111 pages into 317 chunks.

📊 Chunk statistics:
  Average chunk size: 404 characters
  Min chunk size: 40 characters
  Max chunk size: 500 characters

📄 First chunk preview:
------------------------------------------------------------
NetWitness
®
Platform
Version12.5.1.0
NetWitnessUEBAUserGuide
------------------------------------------------------------


# Step 5.1 
- Split the documents into chunks 
- chunk_size=750 and chunk_overlap=200

In [62]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=200,
    add_start_index=True # Helps in tracking exactly where the chunk started
)

all_splits = text_splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} pages into {len(all_splits)} chunks.")
print(f"\n📊 Chunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in all_splits) / len(all_splits):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in all_splits)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in all_splits)} characters")

print(f"\n📄 First chunk preview:")
print("-" * 60)
print(all_splits[0].page_content)
print("-" * 60)

Split 111 pages into 231 chunks.

📊 Chunk statistics:
  Average chunk size: 578 characters
  Min chunk size: 47 characters
  Max chunk size: 749 characters

📄 First chunk preview:
------------------------------------------------------------
NetWitness
®
Platform
Version12.5.1.0
NetWitnessUEBAUserGuide
------------------------------------------------------------


# Step 5.2 
- Split the documents into chunks 
- chunk_size=1000 and chunk_overlap=300

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    add_start_index=True # Helps in tracking exactly where the chunk started
)

all_splits = text_splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} pages into {len(all_splits)} chunks.")
print(f"\n📊 Chunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in all_splits) / len(all_splits):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in all_splits)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in all_splits)} characters")

print(f"\n📄 First chunk preview:")
print("-" * 60)
print(all_splits[0].page_content)
print("-" * 60)

Split 111 pages into 180 chunks.

📊 Chunk statistics:
  Average chunk size: 737 characters
  Min chunk size: 47 characters
  Max chunk size: 1000 characters

📄 First chunk preview:
------------------------------------------------------------
NetWitness
®
Platform
Version12.5.1.0
NetWitnessUEBAUserGuide
------------------------------------------------------------


# Step 6 
- Create Embeddings and Vector Store
- Convert chunks to vector and store in ChromaDB

In [64]:
# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create vector store from chunks
# This will:
# 1. Generate embeddings for all chunks
# 2. Store them in ChromaDB
# 3. Persist to disk for later use

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory="./rag_vector_db"
)

print("✅ Created vector store!")
print(f"📁 Database saved to: ./rag_vector_db")
print(f"📊 Stored {len(all_splits)} document chunks")

✅ Created vector store!
📁 Database saved to: ./rag_vector_db
📊 Stored 180 document chunks


# Step 7 
- Create the RAG Agent

In [65]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0  # Deterministic responses
)

# Create a retrieval tool using the modern @tool decorator pattern
# This follows the latest LangChain documentation pattern
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information from the document store to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=3)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# Create the RAG agent using create_agent (modern pattern)
tools = [retrieve_context]
system_prompt = (
    "You have access to a tool that retrieves context from documents. "
    "Use the tool to help answer user queries. Always cite your sources."
)

rag_agent = create_agent(model, tools, system_prompt=system_prompt)

print("✅ RAG agent created using modern LangChain pattern!")
print("\n📋 Agent configuration:")
print(f"  LLM: {model.model_name}")
print(f"  Retrieval: Top 3 chunks")
print(f"  Pattern: Agent with @tool decorator (latest LangChain docs)")

✅ RAG agent created using modern LangChain pattern!

📋 Agent configuration:
  LLM: gpt-4o-mini
  Retrieval: Top 3 chunks
  Pattern: Agent with @tool decorator (latest LangChain docs)


# Step 8 
- Ask Questions !!!

In [66]:
# Test questions using the modern agent pattern
questions = [
    "How NetWitness UEBA Works ?",
    "How are incidents prioritized ?",
    "What are the UEBA supported sources ?",
    "Explain Forensic Workflow ?",
    "How to identify high risk entities ?"
]

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}\n")
    
    # Use agent.stream() with stream_mode="values" (modern pattern)
    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        # Get the last message from the event
        last_message = event["messages"][-1]
        
        # Display tool calls if present
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print("🔧 Tool Calls:")
            for tool_call in last_message.tool_calls:
                print(f"  - {tool_call.get('name', 'unknown')}: {tool_call.get('args', {})}")
            print()
        
        # Display the final answer
        if hasattr(last_message, 'content') and last_message.content:
            print(f"💬 Answer:\n{last_message.content}\n")
    
    print("-"*70)


❓ Question: How NetWitness UEBA Works ?

💬 Answer:
How NetWitness UEBA Works ?

🔧 Tool Calls:
  - retrieve_context: {'query': 'NetWitness UEBA'}

💬 Answer:
Source: {'source': './documents/nw_12.5.1.0_ueba_user_guide.pdf', 'creator': 'PyPDF', 'page_label': '1', 'page': 0, 'start_index': 0, 'creationdate': '2024-11-19T15:26:59+05:30', 'total_pages': 111, 'keywords': '12.5.1.0pdf', 'producer': 'madbuild', 'subject': '', 'title': 'UEBA User Guide for NetWitness Platform 12.5.1.0', 'moddate': '2024-11-19T15:26:59+05:30', 'author': 'NetWitness Information Design and Development'}
Content: NetWitness
®
Platform
Version12.5.1.0
NetWitnessUEBAUserGuide

Source: {'creator': 'PyPDF', 'keywords': '12.5.1.0pdf', 'start_index': 0, 'producer': 'madbuild', 'moddate': '2024-11-19T15:26:59+05:30', 'author': 'NetWitness Information Design and Development', 'source': './documents/nw_12.5.1.0_ueba_user_guide.pdf', 'page': 0, 'creationdate': '2024-11-19T15:26:59+05:30', 'page_label': '1', 'subject': '', 't

# Note - ❓ Question: How are incidents prioritized ?

For this particular question, the model is picking up the FIA file for answering, despite deleting the original file. Deleting the rag_vector_db folder and trying again to see if it is picking up the UEBA file for analysis.

# Step 9 
- Evaluation & Debugging

In [67]:
def evaluate_retrieval(vectorstore, query, k=3):
    """Evaluate retrieval quality by showing retrieved chunks"""
    results = vectorstore.similarity_search(query, k=k)
    
    print(f"🔍 Query: '{query}'")
    print(f"📊 Retrieved {len(results)} chunks\n")
    
    for i, doc in enumerate(results, 1):
        relevance_score = "✅" if any(keyword in doc.page_content.lower() 
                                     for keyword in query.lower().split()) else "⚠️"
        print(f"{relevance_score} Chunk {i} ({len(doc.page_content)} chars):")
        print(f"   {doc.page_content[:500]}...")
        print()
    
    return results

questions = [
    "Explain briefly about NetWitness UEBA Use Cases ?",
    "Explain alert types ?",
    "What types of charts are available ?",
    "Explain Forensic Workflow ?",
    "How to identify high risk entities ?"
]

# Test retrieval quality
print("="*70)
print("RETRIEVAL QUALITY TEST")
print("="*70)
for question in questions:
    evaluate_retrieval(vectorstore, question, k=3)

RETRIEVAL QUALITY TEST
🔍 Query: 'Explain briefly about NetWitness UEBA Use Cases ?'
📊 Retrieved 3 chunks

✅ Chunk 1 (949 chars):
   NetWitness UEBA Use Cases
NetWitness UEBA focuses on providing advanced detection capabilities to guard enterprises from 
insider threats. These could either be compromised trusted users or network entity within a network, or 
alternatively, an external attacker malicious taking advantage of credentials   acquired by using advanced 
account takeover techniques.
Identity theft typically begins with the theft of credentials, which are then used to obtain unauthorized 
access to resources and to ga...

✅ Chunk 2 (949 chars):
   NetWitness UEBA Use Cases
NetWitness UEBA focuses on providing advanced detection capabilities to guard enterprises from 
insider threats. These could either be compromised trusted users or network entity within a network, or 
alternatively, an external attacker malicious taking advantage of credentials   acquired by using advanced 
ac

# Step 10 
- Create retriever and QA chain

In [68]:
from langchain_classic.chains import RetrievalQA

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [69]:
def evaluate_retrieval(q):
    result = qa_chain.invoke({"query": q})
    
    print(f"\n❓ Question: {q}")
    print(f"💡 Answer: {result['result']}")
    
    print("📚 Sources:")
    for doc in result["source_documents"]:
        print("-", doc.metadata.get("source"), "| page:", doc.metadata.get("page"))

questions = [
    "Explain briefly about NetWitness UEBA Use Cases ?",
    "Explain alert types for a user ?",
    "What types of charts are available ?",
    "Explain Forensic Workflow ?",
    "How to identify high risk entities ?"
]

# Test retrieval quality
print("="*70)
print("RETRIEVAL QUALITY TEST")
print("="*70)
for question in questions:
    evaluate_retrieval(question)

RETRIEVAL QUALITY TEST

❓ Question: Explain briefly about NetWitness UEBA Use Cases ?
💡 Answer: NetWitness UEBA focuses on providing advanced detection capabilities to protect enterprises from insider threats, which can involve compromised trusted users or external attackers using stolen credentials. Key use cases include detecting identity theft that begins with credential theft, unauthorized access to resources, and privilege escalation by attackers exploiting compromised non-admin users. NetWitness UEBA helps differentiate between potentially malicious activities and abnormal but non-risky actions by users or network entities.
📚 Sources:
- ./documents/nw_12.5.1.0_ueba_user_guide.pdf | page: 15
- ./documents/nw_12.5.1.0_ueba_user_guide.pdf | page: 15
- ./documents/nw_12.5.1.0_ueba_user_guide.pdf | page: 15
- ./documents/nw_12.5.1.0_ueba_user_guide.pdf | page: 15

❓ Question: Explain alert types for a user ?
💡 Answer: The alert types for a user include:

1. **Mass Changes to Groups**:

- For each, record:

✅ Did it retrieve the right chunks? - Yes, and mostly the chunks stayed in size limit

✅ Is the answer grounded in the context (not hallucinated)? - The answers are grounded in context. I had tried multiple files to begin with
but had to be bit more specific in my question as the answers contained details from another file (there no UEBA context) that I was using, while 
I was expecting it to answer from UEBA context
 
✅ Yes, answers are mainly correct and relevant o the context. 

Calculate your accuracy: 5/5 retrieval, 5/5 faithfulness, 5/5 correctness (based on RETRIEVAL QUALITY TEST at step 9)

# Step 11

Setting up Ensemble Retriever

In [71]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
)
print("Created fresh vector database!")
print(f"Stored {len(all_splits)} document chunks")

# Create retrievers + ensemble (BM25 + vector similarity)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(all_splits, k=5)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.4, 0.6],
)

print("Created ensemble retriever (vector + BM25)")

Created fresh vector database!
Stored 180 document chunks
Created ensemble retriever (vector + BM25)


# Step 12
Test vector + BM25 ensemble retriever

In [72]:
questions = [
    "Explain briefly about NetWitness UEBA Use Cases ?",
    "Explain alert types for a user ?",
    "What types of charts are available ?"
]

for question in questions:
    results = ensemble_retriever.invoke(question)[:3]

    print("-" * 50)
    print(f"\nTEST QUERY: '{question}'")
    print(f"\nRETRIEVED {len(results)} RELEVANT CHUNKS:")
    print("-" * 50)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. {doc.page_content[:200]}...")
        print(f"   METADATA: {doc.metadata}")
        print("-" * 10)

--------------------------------------------------

TEST QUERY: 'Explain briefly about NetWitness UEBA Use Cases ?'

RETRIEVED 3 RELEVANT CHUNKS:
--------------------------------------------------

1. NetWitness UEBA Use Cases
NetWitness UEBA focuses on providing advanced detection capabilities to guard enterprises from 
insider threats. These could either be compromised trusted users or network en...
   METADATA: {'author': 'NetWitness Information Design and Development', 'page_label': '16', 'creator': 'PyPDF', 'creationdate': '2024-11-19T15:26:59+05:30', 'start_index': 0, 'keywords': '12.5.1.0pdf', 'subject': '', 'title': 'UEBA User Guide for NetWitness Platform 12.5.1.0', 'moddate': '2024-11-19T15:26:59+05:30', 'page': 15, 'producer': 'madbuild', 'total_pages': 111, 'source': './documents/nw_12.5.1.0_ueba_user_guide.pdf'}
----------

2. Contents
 
Introduction 6
Users 6
Network 6
How NetWitness UEBA Works 7
Retrieve Data 8
Create Baselines 8
Create Baselines for Users 8
Create Basel

- Looking at the 3 chunks retirved for each of the questions (weights=[0.5, 0.6])
  
    - Question 1 - Chunk 1 is close to a relevant answer as compared to the other two.
    - Question 2 - Chunk 1 is close to a relevant answer
    - Question 3 - Chunk 1 is close to a relevant answer

- Looking at the 3 chunks retirved for each of the questions (weights=[0.4, 0.6])

    - Question 1 - Chunk 1 is close to a relevant answer as compared to the other two.
    - Question 2 - Chunk 1 is close to a relevant answer
    - Question 3 - Chunk 1 is close to a relevant answer

Same answer as the above iteration.